# Reading CTD files with PyLake

This notebook is for someone who has never used this project. It explains the public reader, each specialized reader, the expected output, and what to do when a real file is unavailable. Run it from the root of the PyLake repository.

## 1. Imports and reproducible example files

The generator creates four tiny deterministic CTD files with 20 measurements. This keeps the tutorial usable without a private dataset or an internet connection.

In [ ]:
from pathlib import Path
import pylake
from examples.generate_ctd_examples import main as generate_examples

data_directory = generate_examples()
sorted(path.name for path in data_directory.iterdir())

## 2. `pylake.read`

`read(path)` is the recommended entry point. It detects DataLakes JSON/NetCDF/ZIP, RBR RSK, KOR CSV, and Sea & Sun TOB files. It returns an `xarray.Dataset`; dimensions describe the axes, variables contain observations, and `attrs['source']` records the detected source.

In [ ]:
datasets = {}
for path in sorted(data_directory.iterdir()):
    datasets[path.name] = pylake.read(path)
    print(path.name, datasets[path.name].attrs["source"], dict(datasets[path.name].sizes))

## 3. `read_datalakes` and `datalakes_to_xarray`

DataLakes stores time in `x`, depth in `y`, and the temperature matrix in `z`. `datalakes_to_xarray` converts an in-memory dictionary; `read_datalakes` opens JSON, NetCDF, or ZIP exports.

In [ ]:
import json

json_path = data_directory / "example_datalakes.json"
payload = json.loads(json_path.read_text())
from_dictionary = pylake.datalakes_to_xarray(payload)
from_file = pylake.read_datalakes(json_path)
from_file

The temperature variable uses `(time, depth)`. Select one profile with `.isel(time=0)` and inspect its physical coordinates.

In [ ]:
from_file.temperature.isel(time=0).to_dataframe().head()

## 4. `read_rsk`

RBR `.rsk` files are SQLite databases. The reader maps channels to dataset variables and preserves units and long names.

In [ ]:
rsk = pylake.read_rsk(data_directory / "example.rsk")
rsk[["temp14", "pres24"]]

## 5. `read_kor`

KOR exports are UTF-16 CSV files with a nine-line header. Numeric columns become dataset variables and the date/time columns become one time coordinate.

In [ ]:
kor = pylake.read_kor(data_directory / "example_kor.csv")
kor

## 6. `read_tob`

Sea & Sun TOB files contain a `; Datasets` marker followed by pressure, temperature, conductivity, chlorophyll, turbidity, pH, and oxygen measurements.

In [ ]:
tob = pylake.read_tob(data_directory / "example.tob")
tob[["pressure", "temperature", "oxygen_mg_l"]]

## 7. Forcing the source

Automatic detection is normally enough. Use `source=` only when an extension is missing or misleading.

In [ ]:
forced = pylake.read(json_path, source="datalakes")
forced.attrs["source"]

## 8. Understand failures

- `FileNotFoundError`: the path is wrong.
- `ValueError` during detection: use a supported file or pass `source=`.
- Shape error in DataLakes: `z` must match time × depth or depth × time.
- Parsing error in KOR/TOB: verify the exporter header and encoding.

When a real file fails, keep the smallest shareable failing sample. If sharing is impossible, regenerate the 20-point examples above and describe exactly how the real file differs.

In [ ]:
try:
    pylake.read("missing-profile.rsk")
except FileNotFoundError as error:
    print(type(error).__name__, error)